# Sampling

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

In [ ]:
# polars
import polars as pl
from IPython.display import display
import numpy as np

def _append(base, extra):
    """Append a value or a Series onto a Series."""
    extra_s = extra if isinstance(extra, pl.Series) else pl.Series([extra])
    if base.len() == 0:
        return extra_s
    if base.dtype != extra_s.dtype:
        try:
            extra_s = extra_s.cast(base.dtype, strict=False)
        except Exception:
            base = base.cast(extra_s.dtype, strict=False)
    return pl.concat([base, extra_s])


## Random Sampling ##

We load in a dataset of all United flights national flights from 6/1/15 to 8/9/15, their destination and how long they were delayed, in minutes.

In [ ]:
# datascience
united = Table.read_table('united.csv')
united = united.with_column('Row', np.arange(united.num_rows)).move_to_start('Row')
united

In [ ]:
# polars
united_pl = pl.read_csv('united.csv', eol_char='\r')
united_pl = (lambda _df, _c='Row': _df.select([_c] + [c for c in _df.columns if c != _c]))(united_pl.with_columns(pl.Series('Row', pl.int_range(united_pl.height, eager=True))))
united_pl

Some deterministic samples:

In [ ]:
# datascience
united.where('Destination', 'JFK') 

In [ ]:
# polars
united_pl.filter(pl.col('Destination') == 'JFK') 

In [ ]:
# datascience
united.take(np.arange(0, united.num_rows, 1000))

In [ ]:
# polars
united_pl.gather(pl.int_range(0, united_pl.height, 1000, eager=True))

In [ ]:
# datascience
united.take(make_array(34, 6321, 10040))

In [ ]:
# polars
united_pl.gather(pl.Series("values", [34, 6321, 10040]))

Random samples:

In [ ]:
# datascience
united.sample(5, with_replacement=False).sort('Row')

In [ ]:
# polars
(united_pl).sample(n=5, with_replacement=False, shuffle=True).sort('Row')

In [ ]:
# datascience
rows = np.random.choice(np.arange(united.num_rows), 5, replace=False)
united.take(rows).sort('Row')

In [ ]:
# polars
rows_pl = (pl.int_range(united_pl.height, eager=True)).sample(5, with_replacement=False)
united_pl.gather(rows_pl).sort('Row')

In [ ]:
# datascience
start = np.random.choice(np.arange(1000))
systematic_sample = united.take(np.arange(start, united.num_rows, 1000))
systematic_sample.show()

In [ ]:
# polars
start_pl = (pl.int_range(1000, eager=True)).sample(1).item()
systematic_sample_pl = united_pl.gather(pl.int_range(start_pl, united_pl.height, 1000, eager=True))
with pl.Config(tbl_rows=-1):
    display(systematic_sample_pl)

## Distributions ##

In [ ]:
# datascience
die = Table().with_column('Face', np.arange(1, 7))
die

In [ ]:
# polars
die_pl = pl.DataFrame({'Face': pl.int_range(1, 7, eager=True)})
die_pl

In [ ]:
# datascience
die.sample(10)

In [ ]:
# polars
(die_pl).sample(n=10, with_replacement=True, shuffle=True)

In [ ]:
# datascience
die.hist('Face')

In [ ]:
# polars
# matplotlib
_df = die_pl
plots.hist(_df.get_column('Face').to_numpy(), density=True)
plots.xlabel('Face')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

In [ ]:
# datascience
roll_bins = np.arange(0.5, 6.6, 1)

In [ ]:
# polars
roll_bins_pl = pl.linear_space(0.5, 6.5, 7, closed="both", eager=True)

In [ ]:
# datascience
die.hist('Face', bins=roll_bins)

In [ ]:
# polars
# matplotlib
_df = die_pl
plots.hist(_df.get_column('Face').to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(roll_bins_pl))
plots.xlabel('Face')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

In [ ]:
# datascience
die.sample(10).hist('Face', bins=roll_bins)

In [ ]:
# polars
# matplotlib
_df = (die_pl).sample(n=10, with_replacement=True, shuffle=True)
plots.hist(_df.get_column('Face').to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(roll_bins_pl))
plots.xlabel('Face')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

In [ ]:
# datascience
die.sample(1000).hist('Face', bins=roll_bins)

In [ ]:
# polars
# matplotlib
_df = (die_pl).sample(n=1000, with_replacement=True, shuffle=True)
plots.hist(_df.get_column('Face').to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(roll_bins_pl))
plots.xlabel('Face')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

In [ ]:
# datascience
die.sample(100000).hist('Face', bins=roll_bins)

In [ ]:
# polars
# matplotlib
_df = (die_pl).sample(n=100000, with_replacement=True, shuffle=True)
plots.hist(_df.get_column('Face').to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(roll_bins_pl))
plots.xlabel('Face')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

## Large Random Samples ##

In [ ]:
# datascience
united 

In [ ]:
# polars
united_pl 

In [ ]:
# datascience
united_bins = np.arange(-20, 201, 5)
united.hist('Delay', bins = united_bins)

In [ ]:
# polars
# matplotlib
united_bins_pl = pl.int_range(-20, 201, 5, eager=True)
_df = united_pl
plots.hist(_df.get_column('Delay').to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(united_bins_pl))
plots.xlabel('Delay')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

In [ ]:
# datascience
min(united.column('Delay'))

In [ ]:
# polars
min(united_pl.get_column('Delay'))

In [ ]:
# datascience
max(united.column('Delay'))

In [ ]:
# polars
max(united_pl.get_column('Delay'))

In [ ]:
# datascience
np.average(united.column('Delay'))

In [ ]:
# polars
(united_pl.get_column('Delay')).mean()

In [ ]:
# datascience
united.sample(10).hist('Delay', bins = united_bins)

In [ ]:
# polars
# matplotlib
_df = (united_pl).sample(n=10, with_replacement=True, shuffle=True)
plots.hist(_df.get_column('Delay').to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(united_bins_pl))
plots.xlabel('Delay')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

In [ ]:
# datascience
united.sample(1000).hist('Delay', bins = united_bins)

In [ ]:
# polars
# matplotlib
_df = (united_pl).sample(n=1000, with_replacement=True, shuffle=True)
plots.hist(_df.get_column('Delay').to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(united_bins_pl))
plots.xlabel('Delay')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

## Simulating Statistics ##

In [ ]:
# datascience
np.median(united.column('Delay'))

In [ ]:
# polars
(united_pl.get_column('Delay')).median()

In [ ]:
# datascience
np.median(united.sample(10).column('Delay'))

In [ ]:
# polars
((united_pl).sample(n=10, with_replacement=True, shuffle=True).get_column('Delay')).median()

In [ ]:
# datascience
def sample_median(size):
    return np.median(united.sample(size).column('Delay'))

In [ ]:
# polars
def sample_median_pl(size):
    return ((united_pl).sample(n=size, with_replacement=True, shuffle=True).get_column('Delay')).median()

In [ ]:
# datascience
sample_median(10)

In [ ]:
# polars
sample_median_pl(10)

In [ ]:
# datascience
sample_medians = make_array()

for i in np.arange(1000):
    new_median = sample_median(10)
    sample_medians = np.append(sample_medians, new_median)

In [ ]:
# polars
sample_medians_pl = []

for i in pl.int_range(1000, eager=True):
    new_median = sample_median_pl(10)
    sample_medians_pl.append(new_median)
sample_medians_pl = pl.Series("values", sample_medians_pl)

In [ ]:
# datascience
Table().with_column('Sample medians', sample_medians).hist(bins = np.arange(-10,31))

In [ ]:
# polars
# matplotlib
_df = pl.DataFrame({'Sample medians': sample_medians_pl})
_cols = [c for c in _df.columns if _df[c].dtype.is_numeric()]
if len(_cols) == 1:
    plots.hist(_df.get_column(_cols[0]).to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(pl.int_range(-10, 31, eager=True)))
    plots.xlabel(_cols[0])
else:
    for _col in _cols:
        plots.hist(_df.get_column(_col).to_numpy(), density=True, label=_col, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(pl.int_range(-10, 31, eager=True)))
    plots.legend()
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))

In [ ]:
# datascience
sample_medians = make_array()

for i in np.arange(1000):
    new_median = sample_median(1000)
    sample_medians = np.append(sample_medians, new_median)

In [ ]:
# polars
sample_medians_pl = []

for i in pl.int_range(1000, eager=True):
    new_median = sample_median_pl(1000)
    sample_medians_pl.append(new_median)
sample_medians_pl = pl.Series("values", sample_medians_pl)

In [ ]:
# datascience
Table().with_column(
    'Sample medians', sample_medians).hist(bins = np.arange(-10,31))

In [ ]:
# polars
# matplotlib
_df = pl.DataFrame({'Sample medians': sample_medians_pl})
_cols = [c for c in _df.columns if _df[c].dtype.is_numeric()]
if len(_cols) == 1:
    plots.hist(_df.get_column(_cols[0]).to_numpy(), density=True, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(pl.int_range(-10, 31, eager=True)))
    plots.xlabel(_cols[0])
else:
    for _col in _cols:
        plots.hist(_df.get_column(_col).to_numpy(), density=True, label=_col, bins=(lambda _b: _b.to_numpy() if hasattr(_b, 'to_numpy') else _b)(pl.int_range(-10, 31, eager=True)))
    plots.legend()
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))